# EGM 722 Project Notebook: Greenspace Analysis (Draft)

## Introduction

Welcome to my project notebook! This will take you through the code required to perform analysis on the amount of Greenspace available in Northern Ireland. 

The code will consist of 4 main parts:
1. **Importing the necessary modules**
2. **Loading the data and setting the CRS**
3. **Carrying out the analysis**
4. **Mapping the results**

**Please Note:** This is currently only a draft - parts may be messy or incomplete!

## Part 1: Preparing Data

To start the project, we first need to import the necessary modules, followed by the data.

In [2]:
#Import the required modules
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from cartopy.feature import ShapelyFeature
import cartopy.crs as ccrs

In [23]:
# Load and name the data layers
greenspace = gpd.read_file(os.path.abspath('data_files/Greenspace_Phase2_06052022.shp'))
lgd = gpd.read_file(os.path.abspath('data_files/LGD.shp'))
dea = gpd.read_file(os.path.abspath('data_files/dea.shp'))
dz = gpd.read_file(os.path.abspath('data_files/DZ2021.shp'))
sdz = gpd.read_file(os.path.abspath('data_files/SDZ2021.shp'))


In [4]:
# View the first 5 rows of the greenspace data to check if it loaded in correctly
greenspace.head(6)

,SourceID,GUID,Name,Source,Category,Type,PaidAccess,Area_Ha,Verified,ShowOnMap,ORNI_ID,DataAdded,SiteCreate,geometry
0,NaN,68d2df20-1512-4218-a759-1000732e93b0,Conservation Volunteers NI,LPS and Outscape,Amenity Greenspace,Community Garden,No,0.668234,Approximated,Yes,51,2022-09-21,Pre 2023,"POLYGON Z ((335088.596 367463.239 0, 334995.26..."
1,NaN,7b20f220-682e-4566-a4fe-2513cc65ed6e,Lough Shore Park,Antrim and Newtownabbey Borough Council,Parks and Gardens,Public Park,No,6.598284,Approximated,Yes,61,2022-09-21,Pre 2023,"POLYGON Z ((313511.357 386601.644 0, 313465.51..."
2,NaN,8a4b8a31-eba9-4e7a-b8d5-cea0724d848d,Ardmore Recreation Centre,"Armagh City, Banbridge and Craigavon Borough C...",Amenity Greenspace,Playing Fields,No,2.969549,Approximated,Yes,67,2022-09-21,Pre 2023,"POLYGON Z ((289345.456 344248.549 0, 289352.83..."
3,NaN,36f2a040-649c-4eb5-9174-7b25c083f489,Clare Glen - Phase 3 - Link Footpath/River Wal...,"Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,3.396109,Approximated,Yes,69,2022-09-21,Pre 2023,"POLYGON Z ((303292.783 345625.097 0, 303291.38..."
4,NaN,2f586e46-b9ee-4d13-83b1-9b254fa3a698,"Folly Glen, Armagh","Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,7.849400,Approximated,Yes,70,2022-09-21,Pre 2023,"MULTIPOLYGON Z (((288701.465 343901.121 0, 288..."
5,NaN,9a79884a-08e5-4789-b0b5-ecdca17688f7,Mossfield Community Scheme,"Armagh City, Banbridge and Craigavon Borough C...",Amenity Greenspace,Public Open Space,No,1.162288,Approximated,Yes,74,2022-09-21,Pre 2023,"POLYGON Z ((297540.433 333393.675 0, 297550.02..."


Now that the data is loaded and looks okay, we will check the crs of each layer, and make any necessary ammendments

In [5]:
# Check the crs of each layer
layer_crs = {'greenspace_crs': [greenspace.crs], 'lgd_crs': [lgd.crs], 'dz_crs': [dz.crs], 'sdz_crs': [sdz.crs]}
crs_table = pd.DataFrame(data=layer_crs)
crs_table

,greenspace_crs,lgd_crs,dz_crs,sdz_crs
0,EPSG:29902,EPSG:29902,EPSG:29902,EPSG:29902


In [6]:
ni_utm = ccrs.UTM(29) # create a Universal Transverse Mercator reference system to transform our data.

In [7]:
ccrs.CRS(greenspace.crs) # create a cartopy CRS representation of the CRS associated with the outline dataset


<Projected CRS: EPSG:29902>
Name: TM65 / Irish Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Ireland - onshore.
- bounds: (-10.56, 51.39, -5.93, 55.43)
Coordinate Operation:
- name: Irish Grid
- method: Transverse Mercator
Datum: TM65
- Ellipsoid: Airy Modified 1849
- Prime Meridian: Greenwich

## Part 2: Greenspace Analysis

This next part of the notebook will demonstrate some analysis using the greenspace data and boundaries provided.

Using the data, we can find:
1. What areas have the highest/lowest coverage of greenspace: at SDZ level and DEA, and per 1000 people.
2. What areas (SDZ) are closest or within a certain distance of greenspaces
3. Bonus: How many parcels are available within each DEA that are available for greenspace? These must be over a certain size, be a natural landscape, within distance of roads, within distance of houses and not already within a greenspace. Which DEAs have the most of these?

So, lets begin...


### Calculating coverage

Before we can calculate what area of greenspace is found within each SDZ, LGD, DEA and so on. It would be useful to calculate the area of each polygon to a standard unit.
This is where our first fucntion will be introduced, one that calculates the area of each polygon, in Km2 and adds it on as a new column to the table.

In [37]:
def area_calc(layer):
    """Caluclates the area of each polygon in the layer and returns a total.
    
    Parameters 
    layer : input polygon layer
    name : name of the layer

    Returns : sum_area_SQKM
        Prints the total area of the polygons in squared kilometers
    """

    layer['Area_SQKM'] = layer['geometry'].area/1e6

    sum_area_SQKM = layer['Area_SQKM'].sum()
    print(f'The total area is {sum_area_SQKM:.2f} kilometers squared')

In [38]:
#test area_calc function on greenspace layer
area_calc(greenspace)

The total area is 875.94 kilometers squared


In [25]:
#test area_calc function column on greenspace layer
greenspace.head(6)

,SourceID,GUID,Name,Source,Category,Type,PaidAccess,Area_Ha,Verified,ShowOnMap,ORNI_ID,DataAdded,SiteCreate,geometry,Area_SQKM
0,NaN,68d2df20-1512-4218-a759-1000732e93b0,Conservation Volunteers NI,LPS and Outscape,Amenity Greenspace,Community Garden,No,0.668234,Approximated,Yes,51,2022-09-21,Pre 2023,"POLYGON Z ((335088.596 367463.239 0, 334995.26...",0.006682
1,NaN,7b20f220-682e-4566-a4fe-2513cc65ed6e,Lough Shore Park,Antrim and Newtownabbey Borough Council,Parks and Gardens,Public Park,No,6.598284,Approximated,Yes,61,2022-09-21,Pre 2023,"POLYGON Z ((313511.357 386601.644 0, 313465.51...",0.065983
2,NaN,8a4b8a31-eba9-4e7a-b8d5-cea0724d848d,Ardmore Recreation Centre,"Armagh City, Banbridge and Craigavon Borough C...",Amenity Greenspace,Playing Fields,No,2.969549,Approximated,Yes,67,2022-09-21,Pre 2023,"POLYGON Z ((289345.456 344248.549 0, 289352.83...",0.029695
3,NaN,36f2a040-649c-4eb5-9174-7b25c083f489,Clare Glen - Phase 3 - Link Footpath/River Wal...,"Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,3.396109,Approximated,Yes,69,2022-09-21,Pre 2023,"POLYGON Z ((303292.783 345625.097 0, 303291.38...",0.033961
4,NaN,2f586e46-b9ee-4d13-83b1-9b254fa3a698,"Folly Glen, Armagh","Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,7.849400,Approximated,Yes,70,2022-09-21,Pre 2023,"MULTIPOLYGON Z (((288701.465 343901.121 0, 288...",0.078494
5,NaN,9a79884a-08e5-4789-b0b5-ecdca17688f7,Mossfield Community Scheme,"Armagh City, Banbridge and Craigavon Borough C...",Amenity Greenspace,Public Open Space,No,1.162288,Approximated,Yes,74,2022-09-21,Pre 2023,"POLYGON Z ((297540.433 333393.675 0, 297550.02...",0.011623


In [21]:
help(area_calc)

Help on function area_calc in module __main__:

area_calc(layer)
    Caluclates the area of each polygon in the layer and returns a total.

    Parameters
    layer : input polygon layer

    Returns : sum_area_SQKM
        Prints the total area of the polygons in squared kilometers



In [39]:
#add a area SQKM column to the remainder of the datasets and find their total area
area_calc(dz)
area_calc(sdz)
area_calc(dea)
area_calc(lgd)

The total area is 13628.31 kilometers squared
The total area is 13628.31 kilometers squared
The total area is 14315.28 kilometers squared
The total area is 14315.28 kilometers squared
